<a href="https://colab.research.google.com/github/diyapatel1225-cmyk/Tomato_Leaf_Disease_Detection-Severity_Assessment/blob/main/Tomato_Disease_Detection_MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tomato Leaf Disease Detection + Severity Assessment (MobileNetV2)

**Project:** AI-Based Tomato Plant Disease Detection and Severity Assessment Using Deep Learning

This notebook walks through the FULL pipeline step by step:
1. Setup & imports
2. Get the dataset (PlantVillage - tomato subset)
3. Preprocessing & augmentation
4. Build MobileNetV2 multi-task model (Disease + Severity)
5. Train (Phase A: frozen base, Phase B: fine-tune)
6. Evaluate (accuracy, F1, confusion matrix)
7. Grad-CAM explainability
8. Save & export model for deployment

> Run cells **top to bottom**. Read the markdown notes above each code cell — they explain *why*, not just *what*.


## Step 1: Setup & Imports

We install/import the libraries we need:
- `tensorflow` / `keras` — the deep learning framework that builds and trains our model
- `numpy`, `pandas` — for handling numbers and tabular data
- `matplotlib`, `seaborn` — for plotting graphs (accuracy curves, confusion matrix)
- `scikit-learn` — for evaluation metrics (precision, recall, F1)
- `opencv-python` — for image handling (used later in Grad-CAM)


In [ ]:
# Run this once. Colab already has TensorFlow, but we pin versions for stability.
!pip install -q tensorflow opencv-python scikit-learn seaborn kaggle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os, shutil, random, zipfile
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


**What just happened:** We loaded all the tools we'll use throughout the notebook.
The GPU check matters — in Colab, go to `Runtime > Change runtime type > GPU (T4)` before running, otherwise
training will be very slow.


## Step 2: Get the Dataset

We use the **PlantVillage tomato subset** — a public dataset with ~11,000+ labeled leaf images across
10 classes (9 diseases + healthy). This is our "textbook" of examples the model will learn from.

**Option A (recommended): Kaggle download**
You need a free Kaggle account and API token (`kaggle.json`).
1. Go to kaggle.com -> your profile -> Account -> "Create New API Token" -> downloads `kaggle.json`
2. Upload that file when prompted below.

**Option B:** If you already have the dataset zipped in Google Drive, mount Drive instead (cell provided below, commented out).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/Tomato_Project/tomato_leaves.zip"

EXTRACT_PATH = "/content/data"
os.makedirs(EXTRACT_PATH, exist_ok=True)

if not os.path.exists(os.path.join(EXTRACT_PATH, "tomato_leaves")):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extracted dataset to", EXTRACT_PATH)
else:
    print("Dataset already extracted, skipping unzip.")

DATA_DIR = "/content/data/tomato/train"
print("Classes found:", os.listdir(DATA_DIR))

Mounted at /content/drive
Extracted dataset to /content/data
Classes found: ['Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Early_blight', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy', 'Tomato___Bacterial_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Septoria_leaf_spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus']


**What just happened:** We now have a folder structure like:

```
data/tomato/train/
    Tomato___Early_blight/
    Tomato___Late_blight/
    Tomato___healthy/
    ...
```

Keras can automatically read this folder structure and treat each subfolder name as a class label —
this is why organizing data this way matters.

### About Severity Labels
PlantVillage does **not** come with Mild/Moderate/Severe labels — only disease names. Since you need
severity too, we generate an approximate severity label automatically using the percentage of diseased
(brown/yellow) pixels in each leaf image. This is a practical workaround many student projects use, and
you should mention this method clearly in your paper's methodology section.


## Step 3: Auto-generate Severity Labels

**How this works:** For diseased leaves, we detect the proportion of unhealthy-colored pixels (brown/yellow/black
spots) versus healthy green pixels using simple color thresholding in HSV color space. More diseased pixels =
higher severity. We bucket the percentage into 4 levels.

This is a rule-based estimate, not perfect — but it's good enough to train a first severity model, and you can
mention in your paper that manual verification on a sample subset was used to validate it.


In [ ]:
def estimate_severity(image_path):
    """Returns severity label: 0=Healthy, 1=Mild, 2=Moderate, 3=Severe"""
    img = cv2.imread(image_path)
    if img is None:
        return 0
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Healthy green range
    green_lower = np.array([25, 40, 40])
    green_upper = np.array([90, 255, 255])
    green_mask = cv2.inRange(hsv, green_lower, green_upper)

    # Diseased (brown/yellow/dark spot) range
    disease_lower = np.array([0, 40, 20])
    disease_upper = np.array([30, 255, 200])
    disease_mask = cv2.inRange(hsv, disease_lower, disease_upper)

    total_leaf_pixels = cv2.countNonZero(green_mask) + cv2.countNonZero(disease_mask)
    if total_leaf_pixels == 0:
        return 0

    diseased_ratio = cv2.countNonZero(disease_mask) / total_leaf_pixels

    if diseased_ratio < 0.05:
        return 0  # Healthy
    elif diseased_ratio < 0.15:
        return 1  # Mild
    elif diseased_ratio < 0.35:
        return 2  # Moderate
    else:
        return 3  # Severe

SEVERITY_NAMES = ["Healthy", "Mild", "Moderate", "Severe"]

# Quick test on one sample image
sample_class = os.listdir(DATA_DIR)[0]
sample_folder = os.path.join(DATA_DIR, sample_class)
sample_img = os.path.join(sample_folder, os.listdir(sample_folder)[0])
print("Sample class:", sample_class)
print("Estimated severity:", SEVERITY_NAMES[estimate_severity(sample_img)])


Sample class: Tomato___Late_blight
Estimated severity: Mild


## Step 4: Build a Labeled DataFrame

We now walk through every image in the dataset, record its file path, disease class, and estimated severity,
all in one table (`pandas DataFrame`). This table becomes the single source of truth that we split into
train/validation/test and feed to the model.


In [ ]:
records = []
for class_name in os.listdir(DATA_DIR):
    class_folder = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_folder):
        continue
    for fname in os.listdir(class_folder):
        fpath = os.path.join(class_folder, fname)
        severity = 0 if "healthy" in class_name.lower() else estimate_severity(fpath)
        records.append({"filepath": fpath, "disease": class_name, "severity": severity})

df = pd.DataFrame(records)
print("Total images:", len(df))
print(df['disease'].value_counts())
print(df['severity'].value_counts())
df.head()


Total images: 10000
disease
Tomato___Late_blight                             1000
Tomato___Leaf_Mold                               1000
Tomato___Early_blight                            1000
Tomato___Tomato_mosaic_virus                     1000
Tomato___healthy                                 1000
Tomato___Bacterial_spot                          1000
Tomato___Spider_mites Two-spotted_spider_mite    1000
Tomato___Target_Spot                             1000
Tomato___Septoria_leaf_spot                      1000
Tomato___Tomato_Yellow_Leaf_Curl_Virus           1000
Name: count, dtype: int64
severity
0    4972
1    2591
2    1283
3    1154
Name: count, dtype: int64


,filepath,disease,severity
0,/content/data/tomato/train/Tomato___Late_bligh...,Tomato___Late_blight,1
1,/content/data/tomato/train/Tomato___Late_bligh...,Tomato___Late_blight,1
2,/content/data/tomato/train/Tomato___Late_bligh...,Tomato___Late_blight,2
3,/content/data/tomato/train/Tomato___Late_bligh...,Tomato___Late_blight,2
4,/content/data/tomato/train/Tomato___Late_bligh...,Tomato___Late_blight,2


**Note:** Running severity estimation on every image can take a few minutes for large datasets.
This is a one-time cost — you can save `df` to a CSV afterward so you don't have to redo it.


In [ ]:
df.to_csv("/content/labeled_dataset.csv", index=False)
print("Saved labeled_dataset.csv with", len(df), "rows")


Saved labeled_dataset.csv with 10000 rows


## Step 5: Encode Labels & Split Data

Neural networks need numeric labels, not text. We convert disease names and severity levels into integers,
then split the data into **train (70%)**, **validation (15%)**, and **test (15%)** sets.

- **Train set:** what the model learns from.
- **Validation set:** checked during training to catch overfitting (model memorizing instead of learning).
- **Test set:** touched only once, at the very end, for the final honest accuracy score you report in your paper.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

disease_encoder = LabelEncoder()
df['disease_label'] = disease_encoder.fit_transform(df['disease'])
NUM_DISEASE_CLASSES = len(disease_encoder.classes_)
NUM_SEVERITY_CLASSES = 4

print("Disease classes:", list(disease_encoder.classes_))

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['disease_label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['disease_label'], random_state=42)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))


Disease classes: ['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
Train: 7000 Val: 1500 Test: 1500


## Step 6: Preprocessing & Augmentation Pipeline

We build a `tf.data` pipeline that, for every image:
1. Reads the file and decodes it as an image
2. Resizes it to 224x224 (the input size MobileNetV2 expects)
3. Normalizes pixel values to the range MobileNetV2 was trained on
4. (Training set only) applies random flips/rotation/brightness changes — this is **augmentation**,
   which teaches the model to handle real-world variation instead of memorizing lab-condition photos

Each image is paired with **two labels** at once — disease and severity — since our model predicts both.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def load_and_preprocess(filepath, disease_label, severity_label, augment=False):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.rot90(img, k=random.randint(0, 3))

    img = keras.applications.mobilenet_v2.preprocess_input(img)
    return img, {"disease_output": disease_label, "severity_output": severity_label}

def make_dataset(dataframe, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        dataframe['filepath'].values,
        dataframe['disease_label'].values,
        dataframe['severity'].values
    ))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    ds = ds.map(lambda f, d, s: load_and_preprocess(f, d, s, augment=augment),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, augment=True, shuffle=True)
val_ds   = make_dataset(val_df, augment=False, shuffle=False)
test_ds  = make_dataset(test_df, augment=False, shuffle=False)

print("Datasets ready.")


Datasets ready.


**What just happened:** We built three ready-to-train pipelines. Notice the training set has
`augment=True` (extra variation) while validation/test don't — we always evaluate on clean, untouched images
so our accuracy numbers are honest.


## Step 7: Build the MobileNetV2 Multi-Task Model

This is the core of the project. Structure:

```
Input Image (224x224x3)
        |
   MobileNetV2 (pretrained on ImageNet, acts as the "eyes")
        |
   Global Average Pooling  <- shrinks feature maps into one vector
        |
   Shared Dense layer
      /        \
Disease Head   Severity Head
 (10 classes)   (4 classes)
```

Both heads branch off the SAME shared features — this is **multi-task learning**. It's efficient (one
backbone instead of two separate models) and often improves accuracy because learning severity and disease
together gives the model extra useful signal.

We start with the MobileNetV2 base **frozen** (its pretrained weights untouched) — this is Phase A from
our earlier explanation.


In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,      # we don't want MobileNet's original 1000-class ImageNet head
    weights='imagenet'      # load the pretrained "eyes"
)
base_model.trainable = False  # Phase A: freeze the base

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
shared = layers.Dense(256, activation='relu', name="shared_dense")(x)
shared = layers.Dropout(0.2)(shared)

# Disease classification head
disease_branch = layers.Dense(128, activation='relu')(shared)
disease_output = layers.Dense(NUM_DISEASE_CLASSES, activation='softmax', name="disease_output")(disease_branch)

# Severity classification head
severity_branch = layers.Dense(64, activation='relu')(shared)
severity_output = layers.Dense(NUM_SEVERITY_CLASSES, activation='softmax', name="severity_output")(severity_branch)

model = keras.Model(inputs=inputs, outputs=[disease_output, severity_output])

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ input_layer_1[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 256)       │    327,936 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ shared_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │     16,448 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ disease_output      │ (None, 10)        │      1,290 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ severity_output     │ (None, 4)         │        260 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,636,814 (10.06 MB)

 Trainable params: 378,830 (1.45 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

**What just happened:** We built the full model architecture. `model.summary()` prints every
layer and how many trainable parameters exist — notice most of MobileNetV2's parameters are frozen
(non-trainable) right now, so only our new head layers will actually learn in Phase A.


## Step 8: Compile the Model

**Compiling** tells Keras three things:
- **Optimizer** (`Adam`): the algorithm that adjusts the model's internal numbers to reduce mistakes
- **Loss functions**: how "wrongness" is measured for each head. Since both outputs are class predictions,
  we use `sparse_categorical_crossentropy` for each.
- **Loss weights**: how much each task's mistake counts toward the total error the model tries to minimize.
  We weight disease slightly higher since it's the primary task.
- **Metrics**: accuracy, tracked for both heads so we can watch training progress.


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "disease_output": "sparse_categorical_crossentropy",
        "severity_output": "sparse_categorical_crossentropy"
    },
    loss_weights={"disease_output": 1.0, "severity_output": 0.7},
    metrics={
        "disease_output": "accuracy",
        "severity_output": "accuracy"
    }
)
print("Model compiled.")


Model compiled.


## Step 9: Phase A Training — Train the New Head Only

We train with the MobileNetV2 base frozen. This is fast and stable: only our new Dense layers are learning,
so there's no risk of accidentally "breaking" the pretrained feature extractor early on.

We use two safety callbacks:
- **EarlyStopping**: stops training automatically if validation performance stops improving, preventing
  wasted time and overfitting.
- **ModelCheckpoint**: saves the best-performing version of the model during training.


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('/content/best_model_phaseA.keras', monitor='val_loss', save_best_only=True)
]

history_a = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)


Epoch 1/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 77s 219ms/step - disease_output_accuracy: 0.6444 - disease_output_loss: 1.0484 - loss: 1.7340 - severity_output_accuracy: 0.5689 - severity_output_loss: 0.9786 - val_disease_output_accuracy: 0.7113 - val_disease_output_loss: 0.8350 - val_loss: 1.4436 - val_severity_output_accuracy: 0.5947 - val_severity_output_loss: 0.8673
Epoch 2/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - disease_output_accuracy: 0.7880 - disease_output_loss: 0.6099 - loss: 1.1951 - severity_output_accuracy: 0.6390 - severity_output_loss: 0.8357 - val_disease_output_accuracy: 0.7487 - val_disease_output_loss: 0.7305 - val_loss: 1.3017 - val_severity_output_accuracy: 0.6187 - val_severity_output_loss: 0.8145
Epoch 3/15
219/219 ━━━━━━━━━━━━━━━━━━━━ 21s 71ms/step - disease_output_accuracy: 0.8129 - disease_output_loss: 0.5494 - loss: 1.1064 - severity_output_accuracy: 0.6483 - severity_output_loss: 0.7960 - val_disease_output_accuracy: 0.7593 - val_disease_output_loss: 0.7224 

## Step 10: Phase B — Fine-Tune MobileNetV2

Now we unfreeze the **last 30 layers** of MobileNetV2 and continue training the whole thing together, but
with a much smaller learning rate. Why smaller? Because the base is already well-trained (on ImageNet) —
large updates now would wreck those useful pretrained weights. Small updates let it gently adapt to
tomato-leaf-specific textures.


In [ ]:
base_model.trainable = True

# Freeze all layers except the last 30 — fine-tune only the deeper, more specialized layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # much smaller LR for fine-tuning
    loss={
        "disease_output": "sparse_categorical_crossentropy",
        "severity_output": "sparse_categorical_crossentropy"
    },
    loss_weights={"disease_output": 1.0, "severity_output": 0.7},
    metrics={
        "disease_output": "accuracy",
        "severity_output": "accuracy"
    }
)

callbacks_b = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('/content/best_model_finetuned.keras', monitor='val_loss', save_best_only=True)
]

history_b = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_b
)


Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 53s 150ms/step - disease_output_accuracy: 0.6234 - disease_output_loss: 1.4444 - loss: 2.3186 - severity_output_accuracy: 0.5123 - severity_output_loss: 1.2471 - val_disease_output_accuracy: 0.7553 - val_disease_output_loss: 0.8486 - val_loss: 1.4105 - val_severity_output_accuracy: 0.6367 - val_severity_output_loss: 0.8008
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 16s 72ms/step - disease_output_accuracy: 0.8157 - disease_output_loss: 0.5561 - loss: 1.1362 - severity_output_accuracy: 0.6387 - severity_output_loss: 0.8283 - val_disease_output_accuracy: 0.7393 - val_disease_output_loss: 0.9740 - val_loss: 1.5424 - val_severity_output_accuracy: 0.6347 - val_severity_output_loss: 0.8097
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - disease_output_accuracy: 0.8493 - disease_output_loss: 0.4340 - loss: 0.9752 - severity_output_accuracy: 0.6643 - severity_output_loss: 0.7725 - val_disease_output_accuracy: 0.7467 - val_disease_output_loss: 0.9256 

## Step 11: Plot Training Curves

These graphs show accuracy/loss over each epoch for both training and validation sets. If training accuracy
keeps rising while validation accuracy flattens or drops, that's overfitting — a key thing to discuss in
your paper's results section.


In [ ]:
def plot_history(history, phase_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['disease_output_accuracy'], label='Train Disease Acc')
    axes[0].plot(history.history['val_disease_output_accuracy'], label='Val Disease Acc')
    axes[0].plot(history.history['severity_output_accuracy'], label='Train Severity Acc')
    axes[0].plot(history.history['val_severity_output_accuracy'], label='Val Severity Acc')
    axes[0].set_title(f'{phase_name}: Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{phase_name}: Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_a, "Phase A (Frozen Base)")
plot_history(history_b, "Phase B (Fine-Tuned)")


## Step 12: Final Evaluation on Test Set

This is the ONE evaluation that matters for your paper's reported accuracy — done on the test set the
model has never seen in any form. We report:
- Overall accuracy for disease and severity
- Per-class precision, recall, F1-score (important because some disease classes have fewer images)
- Confusion matrices (visual — shows exactly which diseases get confused with which)


In [ ]:
# Get true labels and predictions on test set
y_disease_true, y_severity_true = [], []
y_disease_pred, y_severity_pred = [], []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    disease_preds = np.argmax(preds[0], axis=1)
    severity_preds = np.argmax(preds[1], axis=1)

    y_disease_true.extend(labels['disease_output'].numpy())
    y_severity_true.extend(labels['severity_output'].numpy())
    y_disease_pred.extend(disease_preds)
    y_severity_pred.extend(severity_preds)

print("=== DISEASE CLASSIFICATION REPORT ===")
print(classification_report(y_disease_true, y_disease_pred, target_names=disease_encoder.classes_))

print("=== SEVERITY CLASSIFICATION REPORT ===")
print(classification_report(y_severity_true, y_severity_pred, target_names=SEVERITY_NAMES))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

cm_disease = confusion_matrix(y_disease_true, y_disease_pred)
sns.heatmap(cm_disease, annot=True, fmt='d', cmap='Blues',
            xticklabels=disease_encoder.classes_, yticklabels=disease_encoder.classes_, ax=axes[0])
axes[0].set_title('Disease Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

cm_severity = confusion_matrix(y_severity_true, y_severity_pred)
sns.heatmap(cm_severity, annot=True, fmt='d', cmap='Greens',
            xticklabels=SEVERITY_NAMES, yticklabels=SEVERITY_NAMES, ax=axes[1])
axes[1].set_title('Severity Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## Step 13: Grad-CAM — Explainability

Grad-CAM answers: *"Which part of the leaf made the model predict this disease?"* It works by looking at
the last convolutional layer's activations and how strongly each region influenced the final prediction,
then overlays that as a heatmap (red = most influential) on the original image.

This is applied AFTER training — no retraining needed. It's purely for visualization/trust-building.


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_array)
        disease_preds = predictions[0]
        if pred_index is None:
            pred_index = tf.argmax(disease_preds[0])
        class_channel = disease_preds[:, pred_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(img_path, model, last_conv_layer_name="Conv_1", alpha=0.4):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img_array = keras.applications.mobilenet_v2.preprocess_input(np.expand_dims(img.numpy(), axis=0))

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    original = img.numpy().astype('uint8')
    superimposed = cv2.addWeighted(original, 1 - alpha, heatmap_colored, alpha, 0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Grad-CAM: Model's Focus Area")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# Try it on a random test image
sample_path = test_df.iloc[0]['filepath']
print("Actual disease:", test_df.iloc[0]['disease'])
display_gradcam(sample_path, model)


**Note:** `last_conv_layer_name="Conv_1"` is MobileNetV2's final convolutional layer name.
If this errors, run `model.get_layer('mobilenetv2_1.00_224').summary()` (or print `base_model.summary()`)
to find the correct last conv layer name in your TF version, and update it above.


## Step 14: Save & Export for Deployment

We save the trained model in two formats:
- **`.keras`** — full model, for continued use in Python/Colab
- **TFLite** — a compressed, mobile-friendly format for your web/mobile app (matches your project's
  deployment goal and MobileNet's original design purpose)


In [ ]:
# Save full Keras model
model.save('/content/tomato_disease_severity_model.keras')

# Convert to TensorFlow Lite for mobile/web deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # compress for mobile
tflite_model = converter.convert()

with open('/content/tomato_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Saved: tomato_disease_severity_model.keras and tomato_model.tflite")

# Save the class name mappings too - your app will need these to turn predictions back into readable labels
import json
label_map = {
    "disease_classes": list(disease_encoder.classes_),
    "severity_classes": SEVERITY_NAMES
}
with open('/content/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)

print("Saved label_map.json")


## Step 15: (Optional) Download Your Trained Files

Run this to download the trained model and label map to your computer, so you can plug them into a
Flask/FastAPI backend or a mobile app later.


In [ ]:
from google.colab import files
files.download('/content/tomato_disease_severity_model.keras')
files.download('/content/tomato_model.tflite')
files.download('/content/label_map.json')


## What to Report in Your Paper

From this notebook, you now have everything needed for a Methodology + Results section:
- **Dataset**: size, source, class distribution (from Step 4)
- **Preprocessing/Augmentation**: techniques used (Step 6)
- **Architecture**: MobileNetV2 backbone + multi-task heads diagram (Step 7)
- **Training strategy**: two-phase transfer learning (frozen -> fine-tuned) (Steps 9-10)
- **Results**: accuracy, F1-scores, confusion matrices (Step 12)
- **Explainability**: Grad-CAM sample visualizations (Step 13)
- **Deployment**: TFLite export for mobile/web use (Step 14)

### Suggested Next Experiments (for a stronger paper / comparison table)
- Repeat this same pipeline swapping `MobileNetV2` for `EfficientNetB0` and `ResNet50`, then compare
  accuracy/speed/model size in a table — this comparative study is a strong, easy-to-write contribution.
